---

<a id='1'></a>
## Section 3: Data collection
For Section3, we document our code seperate from the main notebook.The RoBERTa model performs a lot of large matrix multiplications when processing text, which is why it runs much faster on a GPU but can be quite slow on a regular CPU environment like many of us have locally. Since that cell mainly runs the sentiment inference step and generates data that we’ve already saved, keeping it in the notebook doesn’t add much value and just slows things down. I think it would be cleaner if we remove that cell and instead attach the preloaded CSV files with the processed results, while keeping the code and data collection scripts in separate files for reference. That way the notebook stays lightweight and everyone can run the analysis much more easily.

---

<a id='1'></a>
## 1. Setup and Configuration

In [3]:
import os, json, re, warnings, logging
import numpy as np
from pathlib import Path

try:
    import requests
    from bs4 import BeautifulSoup
    SCRAPE_AVAILABLE = True
except ImportError:
    print('requests / bs4 not installed. Run: pip install requests beautifulsoup4')
    SCRAPE_AVAILABLE = False
    
np.random.seed(189)

_SKIP_DIRS = {
    'AppData', 'Application Data', 'Local Settings',
    '$Recycle.Bin', 'Windows', 'System Volume Information',
    'Program Files', 'Program Files (x86)', 'ProgramData',
    'node_modules', '.git', '__pycache__',
}

def _find_project_dir():
    for d in [Path.cwd()] + list(Path.cwd().parents):
        if (d / 'data_csv').exists():
            return d
    home = Path.home()
    for root, dirs, _ in os.walk(home):
        root_path = Path(root)
        try:
            depth = len(root_path.relative_to(home).parts)
        except ValueError:
            continue
        dirs[:] = [d for d in dirs if d not in _SKIP_DIRS and not d.startswith('.')]
        if depth >= 6:
            dirs.clear()
            continue
        if (root_path / 'data_csv').exists():
            return root_path
    return Path.cwd()

PROJECT_DIR = Path.cwd()
import tempfile
DATA_DIR = Path(tempfile.gettempdir()) / 'math189_data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')
print('PROJECT_DIR:', PROJECT_DIR)

Setup complete.
PROJECT_DIR: /Users/vikishi/Downloads/math-189-project


In [ ]:
GAMES = {
    # app_id : (game name, lifecycle stratum)
    # --- Popular: sustained high player counts ---
    730:     ('Counter-Strike (CS:GO)',        'popular'),
    570:     ('Dota 2',                        'popular'),
    578080:  ('PUBG: Battlegrounds',           'popular'),
    271590:  ('Grand Theft Auto V',            'popular'),
    230410:  ('Warframe',                      'popular'),
    # --- Declining: peaked pre-2020, downward trend ---
    252490:  ('Rust',                          'decline'),
    346110:  ('ARK: Survival Evolved',         'decline'),
    304390:  ('For Honor',                     'decline'),
    381210:  ('Dead by Daylight',              'decline'),
    359550:  ('Rainbow Six Siege',             'decline'),
    # --- Volatile: event-driven spikes ---
    275850:  ("No Man's Sky",                  'volatile'),
    1085660: ('Destiny 2',                     'volatile'),
    105600:  ('Terraria',                      'volatile'),
    582010:  ('Monster Hunter: World',         'volatile'),
    218620:  ('PAYDAY 2',                      'volatile'),
}

START_DATE  = pd.Timestamp('2020-01-01')
END_DATE    = pd.Timestamp('2025-12-31')
MIN_REPLIES = 100
CSV_DIR     = PROJECT_DIR / 'data_csv'
CSV_DIR.mkdir(parents=True, exist_ok=True)

print(f'Games selected : {len(GAMES)}')
print(f'Date window    : {START_DATE.date()} → {END_DATE.date()}')
print(f'Min replies    : {MIN_REPLIES}')
print()
print(f'{"App ID":>8}  {"Game":<35} {"Stratum"}')
print('-' * 60)
for app_id, (name, stratum) in GAMES.items():
    print(f'{app_id:>8}  {name:<35} {stratum}')

<a id='3.1'></a>
### 3.1 Steam Reviews API Collection

 Steam **rolls up older history into coarse buckets** (often monthly).

To guarantee **one row per week**, we :
1. Use the **GetReviews** endpoint (`store.steampowered.com/appreviews/<appid>?json=1`) with `filter=recent` and **cursor-based pagination** to walk backward in time until we reach `START_DATE`.
2. Run each review's text through a **RoBERTa** sentiment classifier (negative/neutral/positive).
3. Aggregate predictions into weekly counts and **reindex to a complete weekly grid** (`W-MON`) from `START_DATE` to `END_DATE`, filling missing weeks with zeros.

This produces a consistent weekly sentiment panel for every game.

In [ ]:
import re, time
from urllib.parse import quote

# ── Text cleaning ─────────────────────────────────────────────────────────────
def clean_review_text(text):
    """Clean a single review string for sentiment analysis."""
    if not isinstance(text, str):
        return ''
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── GetReviews API: fetch RAW review texts + timestamps back to START_DATE ─────
def fetch_steam_reviews_full(app_id, start_date, language='english', sleep_seconds=0.6, max_reviews=None):
    """
    Stream raw Steam reviews newest→older using cursor pagination until ts < start_date.
    Yields dict rows with keys: timestamp_created, review, voted_up.
    """
    if not SCRAPE_AVAILABLE:
        return

    start_ts = int(pd.Timestamp(start_date, tz='UTC').timestamp())
    url = f"https://store.steampowered.com/appreviews/{app_id}"
    cursor = "*"
    fetched = 0

    headers = {"User-Agent": "Mozilla/5.0 (Math189 Research Project)"}

    while True:
        params = {
            "json": 1,
            "filter": "recent",
            "language": language,
            "review_type": "all",
            "purchase_type": "all",
            "num_per_page": 100,
            "cursor": cursor,
        }
        resp = requests.get(url, params=params, headers=headers, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        reviews = data.get("reviews", [])
        if not reviews:
            break

        cursor = data.get("cursor", cursor)

        for rev in reviews:
            ts = rev.get("timestamp_created")
            if ts is None:
                continue
            if ts < start_ts:
                return

            fetched += 1
            yield {
                "timestamp_created": ts,
                "review": rev.get("review", ""),
                "voted_up": rev.get("voted_up", None),
            }

            if max_reviews is not None and fetched >= max_reviews:
                return

        time.sleep(sleep_seconds)


In [ ]:
# ── Sentiment classification ──────────────────────────────────────────────────
def build_sentiment_pipeline():
    """Load cardiffnlp RoBERTa sentiment model, or return None on failure."""
    if not TRANSFORMERS_AVAILABLE:
        print("  transformers library not available.")
        return None
    try:
        return hf_pipeline(
            "text-classification",
            model="cardiffnlp/twitter-roberta-base-sentiment",
            top_k=None,
            truncation=True,
        )
    except Exception as e:
        print(f"  Sentiment model unavailable: {e}")
        return None


def _pick_label(pred):
    # pred can be dict (top-1) or list[dict] (top_k=None)
    if isinstance(pred, list) and pred:
        pred = max(pred, key=lambda x: x.get("score", 0))
    return str(pred.get("label", "neutral")).lower()


def classify_labels(texts, classifier=None, batch_size=64):
    """Return list of labels in {'negative','neutral','positive'} for each text."""
    if classifier is None:
        # fallback: neutral
        return ["neutral"] * len(texts)

    labels = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        try:
            out = classifier(batch)
        except Exception:
            out = [{"label": "neutral", "score": 1.0}] * len(batch)
        labels.extend([_pick_label(p) for p in out])
    return labels


def weekly_sentiment_from_labels(app_id, dt_series_utc, labels, start_date, end_date):
    """Aggregate predicted labels to weekly counts and reindex to complete W-MON grid."""
    df = pd.DataFrame({"dt": pd.to_datetime(dt_series_utc, utc=True), "label": labels})
    df["week"] = df["dt"].dt.to_period("W-MON").dt.start_time

    weekly = (
        df.groupby("week")["label"]
          .value_counts()
          .unstack(fill_value=0)
          .reset_index()
    )
    for c in ["negative", "neutral", "positive"]:
        if c not in weekly.columns:
            weekly[c] = 0

    weekly["n_posts"] = weekly[["negative", "neutral", "positive"]].sum(axis=1)
    weekly["neg_sentiment"] = weekly["negative"] / weekly["n_posts"].clip(lower=1)
    weekly["pos_sentiment"] = weekly["positive"] / weekly["n_posts"].clip(lower=1)

    weekly = weekly[["week", "neg_sentiment", "pos_sentiment", "n_posts"]].sort_values("week")

    # complete weekly grid
    full_weeks = pd.date_range(
        pd.Timestamp(start_date, tz="UTC").to_period("W-MON").start_time,
        pd.Timestamp(end_date, tz="UTC").to_period("W-MON").start_time,
        freq="W-MON",
    )

    weekly = (
        weekly.set_index("week")
              .reindex(full_weeks)
              .fillna({"neg_sentiment": 0.0, "pos_sentiment": 0.0, "n_posts": 0})
              .rename_axis("week")
              .reset_index()
    )
    weekly["app_id"] = app_id
    return weekly



# MAIN COLLECTION LOOP 
#   1) Pull RAW reviews via GetReviews cursor pagination back to START_DATE
#   2) RoBERTa classify each review
#   3) Aggregate to weekly + reindex to complete weekly grid (W-MON)


CLASSIFIER    = None
sent_records  = {}
review_counts = {}

# Expected number of weekly rows (used for cache validation)
_expected_weeks = len(pd.date_range(
    pd.Timestamp(START_DATE, tz="UTC").to_period("W-MON").start_time,
    pd.Timestamp(END_DATE, tz="UTC").to_period("W-MON").start_time,
    freq="W-MON",
))

for app_id, (name, _) in GAMES.items():
    safe_name = name.replace(":", "").replace("/", "").strip()
    sent_path = CSV_DIR / f"{safe_name}_sentiment.csv"
    if sent_path.exists():
        cached = pd.read_csv(sent_path, parse_dates=["week"])
        if len(cached) >= int(0.9 * _expected_weeks):  # accept small gaps
            sent_records[app_id] = cached
            print(f"  {app_id:>8}  {name:<34}  loaded   {len(cached):>4} weeks")
            continue
        else:
            print(f"  {app_id:>8}  {name:<34}  cache too short ({len(cached)} rows) → rebuilding")

    print(f"  {app_id:>8}  {name:<34}", flush=True)

    # ── Fetch RAW reviews back to START_DATE ──────────────────────────────────
    raw_rows = []
    try:
        for row in fetch_steam_reviews_full(
            app_id,
            start_date=START_DATE,
            language="english",
            sleep_seconds=0.6,
            max_reviews=None,   # set a cap if runtime is too long
        ):
            raw_rows.append(row)
    except Exception as e:
        print(f"           {'':<34}  ⚠ GetReviews fetch failed: {e}")
        raw_rows = []

    if not raw_rows:
        print(f"           {'':<34}  ⚠ no reviews fetched; skipping")
        continue

    rev_df = pd.DataFrame(raw_rows)
    review_counts[app_id] = len(rev_df)
    print(f"           {'':<34}  fetched {len(rev_df):,} reviews")

    # ── Build / load sentiment model once ─────────────────────────────────────
    if CLASSIFIER is None:
        print("           loading RoBERTa sentiment model...")
        CLASSIFIER = build_sentiment_pipeline()

    # ── Classify & aggregate to weekly grid ───────────────────────────────────
    texts = [clean_review_text(t) for t in rev_df["review"].astype(str).tolist()]
    # keep very short empty texts as neutral so alignment stays consistent
    labels = classify_labels(texts, CLASSIFIER, batch_size=64)

    dt = pd.to_datetime(rev_df["timestamp_created"], unit="s", utc=True)
    weekly = weekly_sentiment_from_labels(app_id, dt, labels, START_DATE, END_DATE)

    weekly.to_csv(sent_path, index=False)
    sent_records[app_id] = weekly
    print(f"           {'':<34}  ✓ saved {len(weekly)} weeks to {sent_path.name}")

print(f"\n═══ Sentiment CSVs ready for {len(sent_records)}/{len(GAMES)} games ═══")
if review_counts:
    print(f"Total individual reviews collected: {sum(review_counts.values()):,}")


<a id='3.3'></a>
### 3.3 Player Count Data (SteamCharts)

Historical concurrent player counts come from **SteamCharts** (https://steamcharts.com/). 
SteamCharts records monthly average and peak concurrent player counts for every Steam game, going back many years. 
For each game we scrape the monthly history table and then **interpolate to weekly frequency** 
(using cubic spline interpolation) to align with the weekly sentiment series.

We respect SteamCharts' servers with a 2-second delay between requests.

Player CSVs are written to `data_csv/{GameName}_players.csv` with columns `week`, `players`, `log_players`.


In [ ]:
# ── SteamCharts scraping ──────────────────────────────────────────────────────

def scrape_steamcharts(app_id):
    """
    Scrape monthly player-count history from SteamCharts.
    Returns DataFrame with columns: month, avg_players.
    Returns empty DataFrame on failure.
    """
    if not SCRAPE_AVAILABLE:
        return pd.DataFrame()

    headers = {"User-Agent": "Mozilla/5.0 (Math189 Research Project)"}

    # Try JSON endpoint first
    try:
        url = f"https://steamcharts.com/app/{app_id}/chart-data.json"
        resp = requests.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        rows = [{"month": pd.to_datetime(pt[0], unit="ms"), "avg_players": pt[1]} for pt in data]
        if rows:
            return pd.DataFrame(rows)
    except Exception:
        pass

    # Fallback: scrape the HTML table
    try:
        url_html = f"https://steamcharts.com/app/{app_id}"
        resp = requests.get(url_html, headers=headers, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        table = soup.select_one("table.common-table")
        if table is None:
            return pd.DataFrame()
        rows_data = []
        for tr in table.select("tbody tr"):
            cells = tr.select("td")
            if len(cells) < 5:
                continue
            month_str = cells[0].get_text(strip=True)
            avg_str   = cells[1].get_text(strip=True).replace(",", "")
            try:
                month = pd.to_datetime(month_str, format="%B %Y")
            except Exception:
                try:
                    month = pd.to_datetime(month_str)
                except Exception:
                    continue
            try:
                avg_p = float(re.sub(r"[^\d.]", "", avg_str)) if avg_str else 0
            except ValueError:
                avg_p = 0
            rows_data.append({"month": month, "avg_players": avg_p})
        if rows_data:
            return pd.DataFrame(rows_data)
    except Exception as e:
        print(f"    SteamCharts scrape error: {e}")

    return pd.DataFrame()


def monthly_to_weekly(monthly_df, start="2020-01-06", end="2024-12-30"):
    """Interpolate monthly player data to weekly frequency."""
    df = monthly_df.copy().sort_values("month").set_index("month")
    col = "avg_players" if "avg_players" in df.columns else df.columns[0]
    monthly = df[[col]].rename(columns={col: "players"})
    weekly = monthly.resample("W-MON").interpolate(method="cubic")
    weekly = weekly.loc[start:end].copy()
    weekly["players"] = weekly["players"].clip(lower=1).round().astype(int)
    weekly["log_players"] = np.log(weekly["players"].clip(lower=1))
    weekly = weekly.reset_index().rename(columns={"month": "week"})
    if "week" not in weekly.columns:
        weekly = weekly.reset_index()
        weekly.columns = ["week", "players", "log_players"]
    return weekly


def generate_synthetic_players(app_id, stratum, start="2020-01-06", end="2024-12-30"):
    """Synthetic weekly player counts — ONLY used as a last-resort fallback."""
    rng    = np.random.default_rng(app_id % (2**31))
    weeks  = pd.date_range(start, end, freq="W-MON")
    T      = len(weeks)
    params = {
        'popular':  {'base': 11.5, 'trend':  0.001, 'vol': 0.08},
        'decline':  {'base': 10.0, 'trend': -0.004, 'vol': 0.12},
        'volatile': {'base':  9.5, 'trend': -0.001, 'vol': 0.18},
    }[stratum]
    base  = params["base"]  + rng.normal(0, 0.5)
    trend = params["trend"] + rng.normal(0, 0.001)
    woy   = np.array([w.isocalendar()[1] for w in weeks])
    seas  = 0.15 * np.sin(2 * np.pi * woy / 52) + 0.10 * np.sin(4 * np.pi * woy / 52)
    sale  = ((woy >= 25) & (woy <= 27)) | ((woy >= 51) | (woy <= 1))
    upd   = rng.binomial(1, {"popular": 0.04, "decline": 0.02, "volatile": 0.08}[stratum], T)
    lp    = np.zeros(T)
    lp[0] = base
    for t in range(1, T):
        lp[t] = (0.92 * lp[t-1] + 0.08 * base + trend + seas[t]
                 + 0.15 * upd[t] + 0.08 * sale[t] + rng.normal(0, params["vol"]))
    return pd.DataFrame({
        "week":        pd.to_datetime(weeks),
        "players":     np.exp(lp).round().astype(int),
        "log_players": lp,
        "update_flag": upd.astype(int),
        "season_sale": sale.astype(int),
    })


# ── Load or build player data for every game ─────────────────────────────────
player_records = {}

for app_id, (name, stratum) in GAMES.items():
    safe_name    = name.replace(":", "").replace("/", "").strip()
    player_path  = CSV_DIR / f"{safe_name}_players.csv"

    if player_path.exists():
        df = pd.read_csv(player_path, parse_dates=["week"])
        source = "cached CSV"
    else:
        print(f"  {app_id:>8}  {name:<34}  scraping SteamCharts...", end=" ", flush=True)
        monthly = scrape_steamcharts(app_id)
        time.sleep(2.0)
        if not monthly.empty and len(monthly) >= 6:
            df = monthly_to_weekly(monthly)
            df[["week", "players", "log_players"]].to_csv(player_path, index=False)
            source = f"SteamCharts ({len(monthly)} months)"
        else:
            df = generate_synthetic_players(app_id, stratum)
            df[["week", "players", "log_players"]].to_csv(player_path, index=False)
            source = "synthetic (SteamCharts unavailable)"

    if "update_flag" not in df.columns or "season_sale" not in df.columns:
        syn = generate_synthetic_players(app_id, stratum)
        df = df.merge(syn[["week", "update_flag", "season_sale"]], on="week", how="left")
        df[["update_flag", "season_sale"]] = df[["update_flag", "season_sale"]].fillna(0).astype(int)

    df["app_id"]    = app_id
    df["game"]      = name
    df["safe_name"] = safe_name
    df["stratum"]   = stratum
    player_records[app_id] = df
    print(f"  {app_id:>8}  {name:<34}  {source:>35}  {len(df):>4} weeks")

print(f"\nPlayer CSVs ready for {len(player_records)}/{len(GAMES)} games.")
